Alerta de ordens pendentes criação de KPI:
Ordens pendentes, quantidade de itens e informações de clientes para envio de alerta por email e telefone de quem tem o cadastro completo.

In [0]:
%sql
select * from `2silver_bike`.customers 

In [0]:
%sql
select * from 2silver_bike.orders
where Status='Pending'


In [0]:
describe `2silver_bike`.orders 

In [0]:
SELECT 
O.customer_id,
C.customer_id,
O.Status,
C.email,
C.phone
from 2silver_bike.orders O
inner JOIN 2silver_bike.customers C
on O.customer_id = C.customer_id
WHERE 1=1
AND Status='Pending'

In [0]:
SELECT 
O.customer_id,
O.order_date,
O.store_name,
O.quantity,
O.Status
from 2silver_bike.orders O
WHERE 1=1
AND Status='Pending'

In [0]:
with pending as(SELECT 
customer_id,
order_date,
store_name,
sum(quantity) quantity
from 2silver_bike.orders 
WHERE 1=1
AND upper(Status)='PENDING'
GROUP BY customer_id, order_date, store_name)

SELECT 
P.*,
C.first_name first_name_customer,
C.email,
C.phone
FROM 2silver_bike.customers C
INNER JOIN pending P
ON C.customer_id = P.customer_id


In [0]:
%python
df_orders_pending = spark.sql(
    """
                            
with pending as(
    SELECT 
customer_id,
order_date,
store_name,
sum(quantity) quantity
from 2silver_bike.orders 
WHERE 1=1
AND upper(Status)='PENDING'
GROUP BY customer_id, order_date, store_name)

SELECT 
P.*,
C.first_name first_name_customer,
C.email,
C.phone
FROM 2silver_bike.customers C
INNER JOIN pending P
ON C.customer_id = P.customer_id

                            
                            """
)

display(df_orders_pending)

In [0]:
%python
df_orders_pending.write\
    .mode("overwrite")\
    .format("delta")\
    .option("mergeSchema", "true")\
    .saveAsTable("3gold_bike.orders_pending")

In [0]:
SELECT * FROM 3gold_bike.orders_pending